In [ ]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"
system_prompt = """
あなたはとても簡潔にソリューションを提示できる優秀なエンジニアです。
"""

In [ ]:
# tool
from datetime import datetime

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

# 単にjsonオブジェクトを定義してもよいが、ToolParamでラップするとコードの堅牢性が上がるらしい(?)
from anthropic.types import ToolParam
get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "現在の日時を指定したフォーマットの文字列で取得します。",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "datetime.strftime に渡す日時フォーマット文字列",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})

# ツール名 -> 実行する関数 のマッピング
tool_functions = {
    "get_current_datetime": get_current_datetime,
}

def execute_tool_use_blocks(content_blocks):
    """response.content内のtool_useブロックを実行し、tool_resultブロックのリストを返す"""
    tool_results = []
    for block in content_blocks:
        if block.type == "tool_use":
            result = tool_functions[block.name](**block.input)
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": str(result)
            })
    return tool_results


In [ ]:
# Claude メッセージヘルパー
def to_user_message(text):
    user_message = {"role": "user", "content": text}
    return user_message

def to_assistant_message(content):
    assistant_message = {"role": "assistant", "content": content}
    return assistant_message

def chat(messages, temperature=0.0):
    return client.messages.stream(
        model=model,
        max_tokens=1000,
        temperature=temperature,
        messages=messages,
        system=system_prompt,
        tools=[get_current_datetime_schema]
    )

In [ ]:
messages = []
while(True):
    inputMessage = input("> ")
    if inputMessage == '':
        print('no text input, terminate this program.')
        break

    print(">", inputMessage)
    user_message = to_user_message(inputMessage)
    messages.append(user_message)

    print('---------')
    with chat(messages) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
        response = stream.get_final_message()
    print()
    print('---------')
    assistant_message = to_assistant_message(response.content)
    messages.append(assistant_message)

    # Claudeがツール利用を要求した場合、実行結果を返してから改めて応答を取得する
    while response.stop_reason == "tool_use":
        tool_result_message = {"role": "user", "content": execute_tool_use_blocks(response.content)}
        messages.append(tool_result_message)

        with chat(messages) as stream:
            for text in stream.text_stream:
                print(text, end="", flush=True)
            response = stream.get_final_message()
        print()
        print('---------')
        assistant_message = to_assistant_message(response.content)
        messages.append(assistant_message)